# Task 8: Neural Network Basics with Keras
### CoreTech Client Dataset — Binary Classification

**Steps:**
1. Dummy CoreTech Dataset banao (200+ rows)
2. Preprocessing: train_test_split + StandardScaler
3. Keras Sequential Model (2 hidden layers, ReLU, Sigmoid output)
4. Compile with Adam + binary_crossentropy
5. Train + Plot accuracy/loss curves
6. Evaluate on test data
7. Logistic Regression se compare karo

## Step 0: Libraries Import Karo

In [ ]:
# ============================================================
# Tamam zaroori libraries import kar rahe hain
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Sklearn se preprocessing aur traditional ML model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

# TensorFlow aur Keras import karo
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# TensorFlow ka version check karo
print(f"TensorFlow Version: {tf.__version__}")
print("Tamam libraries successfully import ho gayi hain!")

## Step 1: CoreTech Dummy Dataset Banao

In [ ]:
# ============================================================
# CoreTech ka dummy dataset bana rahe hain
# 250 rows, 6 features, 1 target (project_success: 0 ya 1)
# ============================================================

np.random.seed(42)  # Reproducibility ke liye seed set karo
n = 250             # Total rows ki tadaad

# Features banao
budget          = np.random.randint(50000, 500000, n)        # Project ka budget (PKR)
duration_days   = np.random.randint(30, 365, n)              # Project ki muddat (days)
team_size       = np.random.randint(2, 15, n)                # Team members ki tadaad
client_rating   = np.round(np.random.uniform(1.0, 5.0, n), 1)  # Client ki rating
revisions       = np.random.randint(0, 10, n)                # Kitni baar changes kiye
tech_stack_score = np.random.randint(1, 10, n)               # Technology stack ka score

# Target banao: project_success (0 = fail, 1 = success)
# Realistic logic: budget zyada, team badi, rating achhi ho to success hogi
success_score = (
    (budget > 200000).astype(int) * 2 +
    (team_size > 6).astype(int) * 2 +
    (client_rating > 3.5).astype(int) * 2 +
    (revisions < 5).astype(int) +
    (tech_stack_score > 5).astype(int) +
    (duration_days < 200).astype(int)
)

# Score ke mutabiq success decide karo (threshold = 5)
project_success = (success_score >= 5).astype(int)

# DataFrame banao
df = pd.DataFrame({
    'budget': budget,
    'duration_days': duration_days,
    'team_size': team_size,
    'client_rating': client_rating,
    'revisions': revisions,
    'tech_stack_score': tech_stack_score,
    'project_success': project_success
})

print(f"Dataset Shape: {df.shape}")
print(f"\nTarget Distribution:")
print(df['project_success'].value_counts())
print(f"\nSuccess Rate: {df['project_success'].mean()*100:.1f}%")
print("\nDataset Preview:")
df.head(10)

In [ ]:
# Dataset ki basic statistics dekho
print("Dataset Statistics:")
df.describe()

## Step 2: Preprocessing — Train/Test Split + StandardScaler

In [ ]:
# ============================================================
# Features (X) aur Target (y) alag karo
# ============================================================

X = df.drop('project_success', axis=1)   # Input features
y = df['project_success']                 # Target label

print(f"Features shape: {X.shape}")
print(f"Target shape  : {y.shape}")
print(f"\nFeatures: {list(X.columns)}")

In [ ]:
# ============================================================
# Train aur Test data mein taqseem karo (80% train, 20% test)
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% data test ke liye
    random_state=42,     # Nateeja baar baar ek jaisa aaye
    stratify=y           # Class balance maintain karo
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")

# ============================================================
# StandardScaler se features normalize karo
# Ye zaroori hai taake koi feature doosre par ghalib na ho
# ============================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Train data par fit + transform
X_test_scaled  = scaler.transform(X_test)         # Test data sirf transform karo

print(f"\nScaling complete!")
print(f"Scaled X_train shape: {X_train_scaled.shape}")
print(f"Scaled X_test  shape: {X_test_scaled.shape}")

## Step 3 & 4: Keras Sequential Model Banao + Compile Karo

In [ ]:
# ============================================================
# Keras Sequential Neural Network Model banana
# Architecture:
#   Input Layer  -> 6 features
#   Hidden Layer 1 -> 64 neurons, ReLU activation
#   Hidden Layer 2 -> 32 neurons, ReLU activation
#   Output Layer -> 1 neuron, Sigmoid (binary classification)
# ============================================================

model = Sequential([
    # Pehli hidden layer: 64 neurons, ReLU activation
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),    # Overfitting rokne ke liye 20% neurons randomly band karo

    # Doosri hidden layer: 32 neurons, ReLU activation
    Dense(32, activation='relu'),
    Dropout(0.2),    # Phir se dropout apply karo

    # Output layer: 1 neuron, Sigmoid (0 ya 1 predict karne ke liye)
    Dense(1, activation='sigmoid')
])

# ============================================================
# Model compile karo
# Optimizer: Adam (bahut popular aur effective hai)
# Loss: binary_crossentropy (binary classification ke liye standard)
# Metrics: accuracy track karo training ke dauran
# ============================================================

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Model ka summary dekho
print("Neural Network Architecture:")
print("=" * 55)
model.summary()

## Step 5: Model Train Karo + Accuracy/Loss Curves Plot Karo

In [ ]:
# ============================================================
# EarlyStopping: agar validation loss improve na ho to training rok do
# Ye overfitting se bachata hai
# ============================================================

early_stop = EarlyStopping(
    monitor='val_loss',   # Validation loss dekhte rahenge
    patience=15,          # 15 epochs tak improve na ho to band karo
    restore_best_weights=True  # Best weights wapas laao
)

# ============================================================
# Model train karo
# epochs=100: zyada se zyada 100 baar data dekhe ga
# batch_size=16: har baar 16 samples ek saath process karo
# validation_split=0.2: training data ka 20% validation ke liye
# ============================================================

print("Neural Network Training Shuru Ho Rahi Hai...")
print("=" * 55)

history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print(f"\nTraining {len(history.history['loss'])} epochs mein complete hui!")

In [ ]:
# ============================================================
# Training aur Validation Accuracy/Loss curves plot karo
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Neural Network Training Curves', fontsize=16, fontweight='bold', y=1.02)

epochs_ran = range(1, len(history.history['loss']) + 1)

# --- LEFT: Accuracy Curve ---
axes[0].plot(epochs_ran, history.history['accuracy'],
             color='#2980b9', linewidth=2, label='Training Accuracy')
axes[0].plot(epochs_ran, history.history['val_accuracy'],
             color='#e74c3c', linewidth=2, linestyle='--', label='Validation Accuracy')
axes[0].set_title('Training vs Validation Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epochs', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

# --- RIGHT: Loss Curve ---
axes[1].plot(epochs_ran, history.history['loss'],
             color='#27ae60', linewidth=2, label='Training Loss')
axes[1].plot(epochs_ran, history.history['val_loss'],
             color='#e67e22', linewidth=2, linestyle='--', label='Validation Loss')
axes[1].set_title('Training vs Validation Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epochs', fontsize=11)
axes[1].set_ylabel('Loss', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Curves 'training_curves.png' mein save ho gayi hain!")

## Step 6: Test Data par Evaluate Karo

In [ ]:
# ============================================================
# Neural Network ko test data par evaluate karo
# ============================================================

test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

print("=" * 55)
print("NEURAL NETWORK TEST RESULTS")
print("=" * 55)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")

# Predictions lo
y_pred_prob = model.predict(X_test_scaled, verbose=0)   # Probability values
y_pred_nn   = (y_pred_prob > 0.5).astype(int).flatten() # 0.5 se upar = 1 (success)

print("\nClassification Report (Neural Network):")
print(classification_report(y_test, y_pred_nn, target_names=['Fail (0)', 'Success (1)']))

In [ ]:
# ============================================================
# Confusion Matrix plot karo
# ============================================================

cm_nn = confusion_matrix(y_test, y_pred_nn)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_nn, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fail (0)', 'Success (1)'],
            yticklabels=['Fail (0)', 'Success (1)'],
            linewidths=0.5)
plt.title('Confusion Matrix — Neural Network', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Predicted', fontsize=11)
plt.ylabel('Actual', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrix_nn.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion Matrix 'confusion_matrix_nn.png' mein save ho gayi!")

## Step 7: Logistic Regression — Traditional ML se Compare Karo

In [ ]:
# ============================================================
# Logistic Regression model train karo (Task 5 ki tarah)
# Ye ek traditional ML model hai jise hum Neural Network se compare karein ge
# ============================================================

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)   # Train karo

y_pred_lr = lr_model.predict(X_test_scaled)  # Predictions lo
lr_accuracy = accuracy_score(y_test, y_pred_lr)

print("=" * 55)
print("LOGISTIC REGRESSION TEST RESULTS")
print("=" * 55)
print(f"Test Accuracy : {lr_accuracy * 100:.2f}%")
print("\nClassification Report (Logistic Regression):")
print(classification_report(y_test, y_pred_lr, target_names=['Fail (0)', 'Success (1)']))

In [ ]:
# ============================================================
# Dono models ki accuracy compare karo — bar chart ke zariye
# ============================================================

model_names  = ['Logistic Regression\n(Traditional ML)', 'Neural Network\n(Keras/TensorFlow)']
accuracies   = [lr_accuracy * 100, test_accuracy * 100]
bar_colors   = ['#e67e22', '#2980b9']

plt.figure(figsize=(8, 5))
bars = plt.bar(model_names, accuracies, color=bar_colors,
               edgecolor='black', linewidth=0.8, width=0.45)

# Accuracy values bars ke upar likho
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.5,
             f'{acc:.2f}%',
             ha='center', va='bottom',
             fontsize=14, fontweight='bold')

plt.title('Model Comparison: Logistic Regression vs Neural Network',
          fontsize=13, fontweight='bold', pad=15)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.ylim([0, 115])
plt.xticks(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Comparison chart 'model_comparison.png' mein save ho gaya!")

In [ ]:
# ============================================================
# FINAL SUMMARY — Dono models ka mukammal moazna
# ============================================================

winner = "Neural Network" if test_accuracy > lr_accuracy else "Logistic Regression"
diff   = abs(test_accuracy - lr_accuracy) * 100

print("=" * 60)
print("         TASK 8 — FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"  Dataset Rows              : {len(df)}")
print(f"  Features Used             : {X.shape[1]}")
print(f"  Training Samples          : {X_train_scaled.shape[0]}")
print(f"  Testing  Samples          : {X_test_scaled.shape[0]}")
print("-" * 60)
print(f"  Logistic Regression Acc   : {lr_accuracy*100:.2f}%")
print(f"  Neural Network Accuracy   : {test_accuracy*100:.2f}%")
print("-" * 60)
print(f"  Behtar Model              : {winner}")
print(f"  Farq (Difference)         : {diff:.2f}%")
print("=" * 60)
print("  Tamam Tasks Successfully Complete Ho Gaaye!")
print("=" * 60)